# Day 1 · Section 9: How Qwen Generates Text

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 9.1–9.8

Trace prompt processing, prefill, a KV cache, vocabulary logits, token probabilities, greedy/sampled selection and an autoregressive loop. NumPy/PyTorch exercises run without a model; an optional GPU cell inspects Qwen3-4B's real first-token logits and generated continuation.


In [ ]:
import sys, subprocess, numpy as np
try:
    import torch
    DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',DEVICE)
except ImportError:
    torch=None;DEVICE=None
    print('PyTorch unavailable locally; NumPy path remains runnable. Colab normally includes PyTorch.')
rng=np.random.default_rng(7)


### 9.1–9.4 · A small stateful decoder

This toy decoder uses a cumulative representation to stand in for reusable history. It demonstrates *append one token, update cached state, predict again*. It is not the Key/Value tensors of a real Transformer.


In [ ]:
toy_vocab=['battery','lasts','long','today','<eos>']
toy_vectors=np.eye(len(toy_vocab),dtype=float)
toy_weights=np.array([[.1,2.,.2,0.,0.],[0.,0.,2.,.2,.1],[0.,0.,0.,2.,.1],
                      [0.,0.,0.,0.,3.],[0.,0.,0.,0.,3.]])
def toy_logits(history):
    last=toy_vocab.index(history[-1]);return toy_vectors[last]@toy_weights
history=['battery']
for step in range(5):
    logits=toy_logits(history)
    token=toy_vocab[int(np.argmax(logits))]
    history.append(token)
    print('step',step+1,'appended',token,'history',history)
    if token=='<eos>':break
print('A real KV cache stores layer-specific Keys and Values, not just a word list.')


### 9.5–9.7 · Logits, softmax and decoding controls

One next-token logit per vocabulary item becomes a distribution. Temperature changes sharpness. Top-k and top-p filter candidates *before* sampling; `max_new_tokens` is a length cap.


In [ ]:
def softmax_np(z):
    z=z-np.max(z);e=np.exp(z);return e/e.sum()
vocab=['manual','hood','door','app','moon']
logits=np.array([2.2,1.6,1.1,.2,-.4])
for temperature in [.5,1.,1.5]:
    p=softmax_np(logits/temperature)
    print('temperature',temperature,'p:',np.round(p,3),'greedy:',vocab[int(p.argmax())])
if torch is not None:
    logits_t=torch.tensor(logits,dtype=torch.float32,device=DEVICE)
    p_t=torch.softmax(logits_t,dim=-1)
    assert np.allclose(p_t.cpu().numpy(),softmax_np(logits),atol=1e-6)


In [ ]:
def filtered_distribution(logits,top_k=3,top_p=.8,temperature=1.):
    scaled=logits/temperature
    keep=np.argsort(-scaled)[:top_k]
    p=softmax_np(scaled[keep]);order=np.argsort(-p)
    count=np.searchsorted(np.cumsum(p[order]),top_p,side='left')+1
    selected=keep[order[:count]]
    return selected,softmax_np(scaled[selected])
selected,p=filtered_distribution(logits)
print([(vocab[int(i)],round(float(prob),3)) for i,prob in zip(selected,p)])
assert np.isclose(p.sum(),1.)
rng=np.random.default_rng(7)
print('sampled next token:',vocab[int(rng.choice(selected,p=p))])


### Prefill and KV-cache shapes (conceptual)

Prefill computes prompt positions in parallel while keeping causal attention. During decoding, a new token contributes one new K and V per layer/head. For this toy shape calculation, doubling cached positions doubles stored elements.


In [ ]:
batch,heads,head_dim=1,8,128
for positions in [32,64,128]:
    shape=(batch,heads,positions,head_dim)
    print('one layer K:',shape,'K+V elements:',2*np.prod(shape))
# Queries for earlier positions are not needed for the next-token attention calculation.


### 9.8 · Inspect Qwen3-4B (optional GPU)

The model download is several GB. The default gate requires at least 10 GiB free VRAM. The first forward pass returns logits with shape `[batch, prompt_length, model_vocab_rows]`; the final prompt position predicts the next token.


In [ ]:
import sys, subprocess
try:
    import torch
    HAS_GPU=torch.cuda.is_available()
except ImportError:
    torch=None;HAS_GPU=False
RUN_QWEN=HAS_GPU  # set False to skip the multi-GB download
if HAS_GPU:
    print('GPU:',torch.cuda.get_device_name(0),'free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))
else:print('No CUDA GPU; Qwen3 model cells will be skipped.')


In [ ]:
model=tokenizer=None
if RUN_QWEN and torch.cuda.mem_get_info()[0]/2**30>=10:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.52.4,<6','accelerate','safetensors'])
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model=AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B',torch_dtype=dtype,device_map='auto')
    model.eval()
    print('Loaded model on:',model.device)
else:print('Qwen3 skipped; default gate requires a CUDA GPU with at least 10 GiB free.')


In [ ]:
if model is not None:
    messages=[{'role':'user','content':'Explain RAG in one sentence.'}]
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
        enable_thinking=False,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        first=model(**inputs,use_cache=True)
        next_logits=first.logits[0,-1].float()
        candidates=torch.topk(next_logits,k=5)
        generated=model.generate(**inputs,max_new_tokens=64,do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    print('input IDs:',tuple(inputs['input_ids'].shape),'logits:',tuple(first.logits.shape))
    print('top token IDs:',candidates.indices.cpu().tolist())
    print('first greedy token:',tokenizer.decode([int(candidates.indices[0])]))
    response=generated[0,inputs['input_ids'].shape[1]:]
    print('new tokens:',len(response),'answer:',tokenizer.decode(response,skip_special_tokens=True))
else:print('Qwen3 inspection skipped.')


### Optional manual greedy loop using the model

Run only if the model is loaded. `use_cache=True` lets each decode step reuse prior K/V states. This loop is intentionally short and inspects the first few appended token IDs. The chat template and stop handling are unchanged.


In [ ]:
if model is not None:
    running_ids=inputs['input_ids'];running_mask=inputs.get('attention_mask',torch.ones_like(running_ids))
    past=None;manual=[]
    for step in range(5):
        step_ids=running_ids if past is None else running_ids[:,-1:]
        with torch.inference_mode():
            result=model(input_ids=step_ids,attention_mask=running_mask,
                         past_key_values=past,use_cache=True)
        past=result.past_key_values
        next_id=result.logits[:,-1,:].argmax(dim=-1,keepdim=True)
        manual.append(int(next_id[0,0]))
        running_ids=torch.cat([running_ids,next_id],dim=-1)
        running_mask=torch.cat([running_mask,torch.ones_like(next_id)],dim=-1)
        if int(next_id[0,0])==tokenizer.eos_token_id:break
    print('manual IDs:',manual,'decoded:',tokenizer.decode(manual,skip_special_tokens=True))
else:print('Manual cache loop skipped.')


## Checks

1. Why are prior Keys/Values reusable while a new Query is needed at each step?
2. What is the difference between prompt length and `max_new_tokens`?
3. Does a higher temperature change the greedy winner or only the distribution's sharpness?
4. Compare the manual loop's first token with the first generated token from `generate()`; if they differ, inspect settings and stop handling.
